In [1]:
import pandas as pd
import altair as alt



In [3]:
data = pd.read_csv('cleaned_adult_data.csv')
data.head()

,age,workclass,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income,flagCode,country,DrugUseRateByCountry_2023,DrugUseDALYs_2019
0,39,State-gov,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United States,<=50K,US,United States,3815.24,6.7
1,50,Self-emp-not-inc,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United States,<=50K,US,United States,3815.24,6.7
2,38,Private,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United States,<=50K,US,United States,3815.24,6.7
3,53,Private,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United States,<=50K,US,United States,3815.24,6.7
4,28,Private,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K,CU,Cuba,839.58,1.4


In [10]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 29963 entries, 0 to 31756
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   age                        29963 non-null  int64  
 1   workclass                  29963 non-null  object 
 2   education                  29963 non-null  object 
 3   education-num              29963 non-null  int64  
 4   marital-status             29963 non-null  object 
 5   occupation                 29963 non-null  object 
 6   relationship               29963 non-null  object 
 7   race                       29963 non-null  object 
 8   sex                        29963 non-null  object 
 9   capital-gain               29963 non-null  int64  
 10  capital-loss               29963 non-null  int64  
 11  hours-per-week             29963 non-null  int64  
 12  native-country             29963 non-null  object 
 13  income                     29963 non-null  object 


In [4]:
relationship_income = data.groupby('marital-status').agg({
    'income': lambda x: (x == '>50K').sum() / len(x)
}).reset_index()
relationship_income.columns = ['marital-status', 'high_income_rate']
relationship_income = relationship_income.sort_values('high_income_rate', ascending=False)

chart = alt.Chart(relationship_income).mark_bar().encode(
    x=alt.X('high_income_rate:Q', title='Proportion with Income >50K'),
    y=alt.Y('marital-status:N', title='Marital Status', sort='-x'),
).properties(
    width=600,
    height=300,
    title='Income Distribution by Marital Status'
)

chart.show()

alt.Chart(...)

In [5]:

data = data[data['workclass'] != '?']

workclass_education = data.groupby(['workclass', 'education']).size().reset_index(name='count')
workclass_education['total'] = workclass_education.groupby('workclass')['count'].transform('sum')
workclass_education['percentage'] = (workclass_education['count'] / workclass_education['total'] * 100).round(2)

chart = alt.Chart(workclass_education).mark_bar().encode(
    x=alt.X('workclass:N', title='Work Class'),
    y=alt.Y('percentage:Q', title='Percentage (%)', stack='normalize'),
    color=alt.Color('education:N', title='Education Level'),
    tooltip=['workclass:N', 'education:N', 'percentage:Q']
).properties(
    width=800,
    height=400,
    title='Education Level Distribution by Workclass (%)'
)

chart.show()

alt.Chart(...)

In [6]:

race_income = data.groupby('race').agg({
    'income': lambda x: (x == '>50K').sum() / len(x)
}).reset_index()
race_income.columns = ['race', 'high_income_rate']
race_income = race_income.sort_values('high_income_rate', ascending=False)


chart = alt.Chart(race_income).mark_bar().encode(
    x=alt.X('high_income_rate:Q', title='Proportion with Income >50K'),
    y=alt.Y('race:N', title='Race', sort='-x'),

).properties(
    width=600,
    height=300,
    title='Income Distribution by Race'
)

chart.show()

alt.Chart(...)

In [7]:
education_income = data.groupby('education').agg({
    'income': lambda x: (x == '>50K').sum() / len(x)
}).reset_index()
education_income.columns = ['education', 'high_income_rate']
education_income = education_income.sort_values('high_income_rate', ascending=False)

chart = alt.Chart(education_income).mark_bar().encode(
    x=alt.X('high_income_rate:Q', title='Proportion with Income >50K'),
    y=alt.Y('education:N', title='Education Level', sort='-x'),
).properties(
    width=600,
    height=400,
    title='Income Distribution by Education Level'
)

chart.show()


alt.Chart(...)

In [8]:
workclass_income = data.groupby('workclass').agg({
    'income': lambda x: (x == '>50K').sum() / len(x)
}).reset_index()
workclass_income.columns = ['workclass', 'high_income_rate']
workclass_income = workclass_income.sort_values('high_income_rate', ascending=False)

chart = alt.Chart(workclass_income).mark_bar().encode(
    x=alt.X('high_income_rate:Q', title='Proportion with Income >50K'),
    y=alt.Y('workclass:N', title='Work Class', sort='-x'),
).properties(
    width=600,
    height=400,
    title='Income Distribution by Work Class'
)

chart.show()


alt.Chart(...)

In [9]:
sex_income = data.groupby('sex').agg({
    'income': lambda x: (x == '>50K').sum() / len(x)
}).reset_index()
sex_income.columns = ['sex', 'high_income_rate']
sex_income = sex_income.sort_values('high_income_rate', ascending=False)

chart = alt.Chart(sex_income).mark_bar().encode(
    x=alt.X('high_income_rate:Q', title='Proportion with Income >50K'),
    y=alt.Y('sex:N', title='Sex', sort='-x'),
).properties(
    width=600,
    height=300,
    title='Income Distribution by Sex'
)

chart.show()


alt.Chart(...)

In [12]:
import xgboost as xgb
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

In [25]:
df = data.copy()
df['target'] = (df['income'] == '<=50K').astype(int)
features = [c for c in df.columns if c not in ['income', 'target']]
for col in df[features].select_dtypes(include='object').columns:
    df[col] = LabelEncoder().fit_transform(df[col])

In [26]:
df['target']

0        1
1        1
2        1
3        1
4        1
        ..
31752    1
31753    0
31754    1
31755    1
31756    0
Name: target, Length: 29963, dtype: int64

In [27]:
X, y = df[features], df['target']

In [28]:
selected = []
best_acc = 0

In [29]:
for round in range(len(features)):
    results = {}
    for f in features:
        if f not in selected:
            cols = selected + [f]
            model = xgb.XGBClassifier(n_estimators=100, max_depth=5, verbosity=0, random_state=42)
            acc = cross_val_score(model, X[cols], y, cv=5, scoring='accuracy').mean()
            results[f] = acc
            print("Testing", f, ' - ', acc)
    
    best_feat = max(results, key=results.get)
    if results[best_feat] > best_acc:
        best_acc = results[best_feat]
        selected.append(best_feat)
        print("Round ",  round+1, ": Added '", best_feat, " Accuracy: ", best_acc)
    else:
        print("Round ",  round+1, ": No improvement. Stopping.")
        break

print("Final features (", len(selected), "): ", selected)
print("Final accuracy: ", best_acc)


Testing age  -  0.7502920184808399
Testing workclass  -  0.7547976143506988
Testing education  -  0.7727531474749023
Testing education-num  -  0.7727531474749023
Testing marital-status  -  0.7503587852940135
Testing occupation  -  0.7504922743645958
Testing relationship  -  0.7489570943581931
Testing race  -  0.7504922743645958
Testing sex  -  0.7504922743645958
Testing capital-gain  -  0.8028902656125071
Testing capital-loss  -  0.7723524686232738
Testing hours-per-week  -  0.7502920129113694
Testing native-country  -  0.7504922743645958
Testing flagCode  -  0.7504922743645958
Testing country  -  0.7504922743645958
Testing DrugUseRateByCountry_2023  -  0.7504922743645958
Testing DrugUseDALYs_2019  -  0.7504922743645958
Round  1 : Added ' capital-gain  Accuracy:  0.8028902656125071
Testing age  -  0.8025231428210526
Testing workclass  -  0.8027901488095702
Testing education  -  0.8133031427185745
Testing education-num  -  0.8133365149862201
Testing marital-status  -  0.8027567765419248

In [31]:
selected

['capital-gain',
 'capital-loss',
 'education',
 'relationship',
 'occupation',
 'age',
 'workclass',
 'hours-per-week',
 'DrugUseRateByCountry_2023']

WE can clearly see the features that play a part in income prediction above